24520002 - Mai Quốc Anh

24520010 - Đặng Phú Duy

24521703 - Phạm Ngọc Phú Thịnh

In [10]:
!gdown 1S8abCHVS9COuWhPDwBI-xtI1SnnPmBgA

Downloading...
From: https://drive.google.com/uc?id=1S8abCHVS9COuWhPDwBI-xtI1SnnPmBgA
To: /content/items.parquet
100% 2.90M/2.90M [00:00<00:00, 16.3MB/s]


In [11]:
!gdown 1LizMhLDpMSYRoog6HRXi0n-kGihzaLPz

Downloading...
From: https://drive.google.com/uc?id=1LizMhLDpMSYRoog6HRXi0n-kGihzaLPz
To: /content/transactions-2025-12.parquet
100% 48.2M/48.2M [00:01<00:00, 42.1MB/s]


In [12]:
import polars as pl

df = pl.read_parquet("items.parquet")
df_trans = pl.read_parquet("transactions-2025-12.parquet")
print(df.columns)

['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'brand', 'manufacturer', 'description', 'sale_status', 'size']


In [13]:
import polars as pl

item_df = pl.read_parquet(r"items.parquet")

result = item_df.with_columns([
    # step: 1-5 -> int
    pl.when(
        pl.col("description").str.contains("(?i)cho mẹ bầu")
        & pl.col("description").str.extract(r"step\s*([1-5])", 1).is_null()
    )
    .then(0)
    .otherwise(
        pl.col("description")
        .str.extract(r"step\s*([1-5])", 1)
        .cast(pl.Int8)
    )
    .alias("step"),

    # size -> map thành số
    pl.col("description")
    .str.extract(r"\b(Newborn|S|M|L|XL|XXL)\b", 1)
    .replace({
        "Newborn": 0,
        "S": 1,
        "M": 2,
        "L": 3,
        "XL": 4,
        "XXL": 5
    })
    .cast(pl.Int8)
    .alias("size"),

    # piece -> số
    pl.col("description")
    .str.extract(r"(\d+)\s*miếng", 1)
    .cast(pl.Int16)
    .alias("piece"),
]).select([
    "item_id",
    "category_l1",
    "category_l2",
    "category_l3",
    "category",
    "step",
    "size",
    "piece",
    "description"
])

print(result['size'].unique())
print(result['step'].unique())
print(result['piece'].unique())


shape: (7,)
Series: 'size' [i8]
[
	null
	0
	1
	2
	3
	4
	5
]
shape: (7,)
Series: 'step' [i8]
[
	null
	0
	1
	2
	3
	4
	5
]
shape: (54,)
Series: 'piece' [i16]
[
	null
	1
	2
	3
	4
	…
	84
	90
	100
	108
	200
]


In [14]:
print(result)

shape: (29_823, 9)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬──────┬──────┬───────┬─────────────┐
│ item_id      ┆ category_l1  ┆ category_l2  ┆ category_l3 ┆ … ┆ step ┆ size ┆ piece ┆ description │
│ ---          ┆ ---          ┆ ---          ┆ ---         ┆   ┆ ---  ┆ ---  ┆ ---   ┆ ---         │
│ str          ┆ str          ┆ str          ┆ str         ┆   ┆ i8   ┆ i8   ┆ i16   ┆ str         │
╞══════════════╪══════════════╪══════════════╪═════════════╪═══╪══════╪══════╪═══════╪═════════════╡
│ 000804000004 ┆ Đồ chơi &    ┆ 1Y+          ┆ Học tập và  ┆ … ┆ null ┆ null ┆ null  ┆ Robo Luồn   │
│ 6            ┆ Sách         ┆              ┆ phát triển  ┆   ┆      ┆      ┆       ┆ thun Winwin │
│              ┆              ┆              ┆ tư duy      ┆   ┆      ┆      ┆       ┆ toys có h…  │
│ 050202000000 ┆ Babycare     ┆ Bình sữa,    ┆ Núm ty      ┆ … ┆ null ┆ null ┆ null  ┆ Không xác   │
│ 4            ┆              ┆ phụ kiện     ┆             ┆   ┆      ┆ 

In [15]:
import math

def get_pair_co_buy_score(item_x, item_a):
    # 1. Lấy danh sách các phiên mua hàng (duy nhất) của item_x
    sessions_x = (
        df_trans.filter(pl.col("item_id") == item_x)
        .select(["customer_id", "updated_date"])
        .unique()
    )

    # 2. Lấy danh sách các phiên mua hàng (duy nhất) của item_a
    sessions_a = (
        df_trans.filter(pl.col("item_id") == item_a)
        .select(["customer_id", "updated_date"])
        .unique()
    )

    # 3. Inner join để tìm các phiên mà người dùng mua CẢ 2 sản phẩm
    common_sessions = sessions_x.join(sessions_a, on=["customer_id", "updated_date"], how="inner")

    # 4. Trả về số lượng dòng (chính là co_buy_score)
    return common_sessions.height
def compute_cat_score(itemX, itemA):
    """
    itemX, itemA: dict chứa các field:
        - category
        - category_l3
        - category_l2
        - category_l1
    """

    # Ưu tiên match từ level chi tiết nhất → tổng quát nhất
    if itemX.get("category") == itemA.get("category"):
        return 1.0
    elif itemX.get("category_l3") == itemA.get("category_l3"):
        return 0.5
    elif itemX.get("category_l2") == itemA.get("category_l2"):
        return 0.25
    elif itemX.get("category_l1") == itemA.get("category_l1"):
        return 0.125
    else:
        return 0.0

def compute_size_score(itemX, itemA, W1=1.5, W2=0.5):
    """
    itemX, itemA: dict chứa key 'size'
    """

    size_X = itemX.get("size")
    size_A = itemA.get("size")

    # handle missing value
    if size_X is None or size_A is None:
        return 1

    diff_XA = max(0, size_X - size_A)
    diff_AX = max(0, size_A - size_X)

    score = math.exp(-W1 * diff_XA - W2 * diff_AX)

    return score

def compute_step_score(itemX, itemA, W1=1.5, W2=0.5):
    """
    itemX, itemA: dict chứa key 'step'
    """

    step_X = itemX.get("step")
    step_A = itemA.get("step")

    # handle missing
    if step_X is None or step_A is None:
        return 1

    try:
        step_X = float(step_X)
        step_A = float(step_A)
    except:
        return 1

    diff_XA = max(0.0, step_X - step_A)
    diff_AX = max(0.0, step_A - step_X)

    return math.exp(-W1 * diff_XA - W2 * diff_AX)


def compute_piece_score(itemX, itemA, W1=1.5, W2=0.5):
    """
    itemX, itemA: dict chứa key 'piece'
    """

    piece_X = itemX.get("piece")
    piece_A = itemA.get("piece")

    # handle missing
    if piece_X is None or piece_A is None:
        return 0.0

    try:
        piece_X = float(piece_X)
        piece_A = float(piece_A)
    except:
        return 0.0

    diff_XA = max(0.0, piece_X - piece_A)
    diff_AX = max(0.0, piece_A - piece_X)

    return math.exp(-W1 * diff_XA - W2 * diff_AX)




In [16]:
def compute_score(itemX, itemA):
    """
    item_X, item_A: dict chứa thông tin item
        {
            "id": ...,
            "category": ...,
            "price": ...
        }
    """

    # 1. co-buy score
    co_buy = get_pair_co_buy_score(itemX['item_id'], itemA['item_id'])

    # 2. category score
    cat_score = compute_cat_score(itemX, itemA)

    # 3. price score
    size_score = compute_size_score(itemX, itemA)

    # 4. step score
    step_score = compute_step_score(itemX, itemA)

    # 5. piece score
    piece_score = compute_piece_score(itemX, itemA)

    # final score
    final_score = co_buy * cat_score * size_score * step_score

    return final_score

In [17]:
def recommend_items(idX, df):
    # lấy đúng 1 row của itemX
    rowX = df.filter(df["item_id"] == idX)

    if rowX.height == 0:
        raise ValueError("itemX không tồn tại")

    itemX = rowX.to_dicts()[0]  # convert thành dict

    results = []
    i = 0
    for row in df.iter_rows(named=True):
        # bỏ chính nó
        i += 1
        # print(f"process data {i}")
        if row["item_id"] == idX:
            continue

        itemA = dict(row)

        score = compute_score(itemX, itemA)

        results.append({
            "item_id": itemA["item_id"],
            "score": score,
            "category" : itemA["category"],
            "description": itemA["description"]
        })

    # sort giảm dần
    results.sort(key=lambda x: x["score"], reverse=True)

    return results

In [ ]:
rec = recommend_items(3389000000238, result)
print(rec)